In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [4]:
# Sample data
data = [('James', 'Sales', 'NY', 9000, 34),
        ('Alicia', 'Sales', 'NY', 8600, 56),
        ('Robert', 'Sales', 'CA', 8100, 30),
        ('John', 'Sales', 'AZ', 8600, 31),
        ('Ross', 'Sales', 'AZ', 8100, 33),
        ('Kathy', 'Sales', 'AZ', 1000, 39),
        ('Lisa', 'Finance', 'CA', 9000, 24),
        ('Deja', 'Finance', 'CA', 9900, 40),
        ('Sugie', 'Finance', 'NY', 8300, 36),
        ('Ram', 'Finance', 'NY', 7900, 53),
        ('Satya', 'Finance', 'AZ', 8200, 53),
        ('Kyle', 'Marketing', 'CA', 8000, 25),
        ('Reid', 'Marketing', 'NY', 9100, 50)]
df = spark.createDataFrame(data, ['Name', 'Department', 'State', 'Salary', 'Age'])
df.show()

+------+----------+-----+------+---+
|  Name|Department|State|Salary|Age|
+------+----------+-----+------+---+
| James|     Sales|   NY|  9000| 34|
|Alicia|     Sales|   NY|  8600| 56|
|Robert|     Sales|   CA|  8100| 30|
|  John|     Sales|   AZ|  8600| 31|
|  Ross|     Sales|   AZ|  8100| 33|
| Kathy|     Sales|   AZ|  1000| 39|
|  Lisa|   Finance|   CA|  9000| 24|
|  Deja|   Finance|   CA|  9900| 40|
| Sugie|   Finance|   NY|  8300| 36|
|   Ram|   Finance|   NY|  7900| 53|
| Satya|   Finance|   AZ|  8200| 53|
|  Kyle| Marketing|   CA|  8000| 25|
|  Reid| Marketing|   NY|  9100| 50|
+------+----------+-----+------+---+



In [5]:
from pyspark.sql.window import *
from pyspark.sql.functions import *

## Ranking Window Functions
1. row_number(): Assigns unique sequential integer to each row within partition
   - No ties, always unique numbers

2. rank(): Assigns rank with gaps for ties
   - Same values get same rank, next rank skips numbers

3. dense_rank(): Assigns rank without gaps for ties
   - Same values get same rank, next rank continues sequentially

4. percent_rank(): Returns percentile rank (0.0 to 1.0)
   - Formula: (rank - 1) / (total rows - 1)

5. ntile(n): Divides rows into n buckets
   - Useful for creating quartiles, deciles, etc.

6. cume_dist(): Returns cumulative distribution (0.0 to 1.0)
   - Formula: the fraction of rows that are below the current row.

In [7]:
# Define the window spec object
windowSpec = Window.partitionBy(df.Department).orderBy(df.Salary.desc())
windowSpec

In [8]:
result_df = df.select(df.Department, df.Salary).withColumn("row_number", row_number().over(windowSpec)) \
    .withColumn("rank", rank().over(windowSpec)) \
    .withColumn("dense_rank", dense_rank().over(windowSpec)) \
    .withColumn("percent_rank", percent_rank().over(windowSpec)) \
    .withColumn("ntile_3", ntile(3).over(windowSpec)) \
    .withColumn("cume_dist", cume_dist().over(windowSpec))

result_df.show(truncate=False)

+----------+------+----------+----+----------+------------+-------+-------------------+
|Department|Salary|row_number|rank|dense_rank|percent_rank|ntile_3|cume_dist          |
+----------+------+----------+----+----------+------------+-------+-------------------+
|Finance   |9900  |1         |1   |1         |0.0         |1      |0.2                |
|Finance   |9000  |2         |2   |2         |0.25        |1      |0.4                |
|Finance   |8300  |3         |3   |3         |0.5         |2      |0.6                |
|Finance   |8200  |4         |4   |4         |0.75        |2      |0.8                |
|Finance   |7900  |5         |5   |5         |1.0         |3      |1.0                |
|Marketing |9100  |1         |1   |1         |0.0         |1      |0.5                |
|Marketing |8000  |2         |2   |2         |1.0         |2      |1.0                |
|Sales     |9000  |1         |1   |1         |0.0         |1      |0.16666666666666666|
|Sales     |8600  |2         |2 

## Analytical Window Functions
- lag()
- lead()

In [9]:
# NULL is poupulated if offset record not present
result_df = df.select(df.Department, df.Salary).withColumn('lag_salary', lag(df.Salary, 1).over(windowSpec)).withColumn('lead_salary', lead(df.Salary, 1).over(windowSpec))
result_df.show(truncate=False)

+----------+------+----------+-----------+
|Department|Salary|lag_salary|lead_salary|
+----------+------+----------+-----------+
|Finance   |9900  |NULL      |9000       |
|Finance   |9000  |9900      |8300       |
|Finance   |8300  |9000      |8200       |
|Finance   |8200  |8300      |7900       |
|Finance   |7900  |8200      |NULL       |
|Marketing |9100  |NULL      |8000       |
|Marketing |8000  |9100      |NULL       |
|Sales     |9000  |NULL      |8600       |
|Sales     |8600  |9000      |8600       |
|Sales     |8600  |8600      |8100       |
|Sales     |8100  |8600      |8100       |
|Sales     |8100  |8100      |1000       |
|Sales     |1000  |8100      |NULL       |
+----------+------+----------+-----------+



In [10]:
# With default value argument
result_df = df.select(df.Department, df.Salary).withColumn('lag_salary_with_default', lag(df.Salary, 1, 0).over(windowSpec)).withColumn('lead_salary_with_default', lead(df.Salary, 1, -1).over(windowSpec))
result_df.show(truncate=False)

+----------+------+-----------------------+------------------------+
|Department|Salary|lag_salary_with_default|lead_salary_with_default|
+----------+------+-----------------------+------------------------+
|Finance   |9900  |0                      |9000                    |
|Finance   |9000  |9900                   |8300                    |
|Finance   |8300  |9000                   |8200                    |
|Finance   |8200  |8300                   |7900                    |
|Finance   |7900  |8200                   |-1                      |
|Marketing |9100  |0                      |8000                    |
|Marketing |8000  |9100                   |-1                      |
|Sales     |9000  |0                      |8600                    |
|Sales     |8600  |9000                   |8600                    |
|Sales     |8600  |8600                   |8100                    |
|Sales     |8100  |8600                   |8100                    |
|Sales     |8100  |8100           

## Aggregate Window Functions

In [11]:
result_df = df.select(df.Department, df.Salary).withColumn('cumsum_salary', sum(df.Salary).over(windowSpec)) \
  .withColumn('avg_salary', avg(df.Salary).over(windowSpec)) \
  .withColumn('max_salary', max(df.Salary).over(windowSpec)) \
  .withColumn('min_salary', min(df.Salary).over(windowSpec)) \
  .withColumn('count_dept', count(df.Salary).over(windowSpec)) \
  .withColumn('first_salary', first(df.Salary).over(windowSpec)) \
  .withColumn('last_salary', last(df.Salary).over(windowSpec))

result_df.show()

+----------+------+-------------+-----------------+----------+----------+----------+------------+-----------+
|Department|Salary|cumsum_salary|       avg_salary|max_salary|min_salary|count_dept|first_salary|last_salary|
+----------+------+-------------+-----------------+----------+----------+----------+------------+-----------+
|   Finance|  9900|         9900|           9900.0|      9900|      9900|         1|        9900|       9900|
|   Finance|  9000|        18900|           9450.0|      9900|      9000|         2|        9900|       9000|
|   Finance|  8300|        27200|9066.666666666666|      9900|      8300|         3|        9900|       8300|
|   Finance|  8200|        35400|           8850.0|      9900|      8200|         4|        9900|       8200|
|   Finance|  7900|        43300|           8660.0|      9900|      7900|         5|        9900|       7900|
| Marketing|  9100|         9100|           9100.0|      9100|      9100|         1|        9100|       9100|
| Marketin

#### rowsBetween(start, end)
- `rowsBetween` is used to define the frame of rows in a window function by specifying a start and end row relative to the current row.
- Example: `rowsBetween(2, 2)` will create a window staring from 2 rows before the current row and 2 rows after the current row, so total 5 rows

#### rangeBetween(start, end)
- `rangeBetween` is similar to `rowsBetween` but operates on ranges rather than specific row counts.
- It considers the values of the rows rather than their positions.
- Example: If the current salary is 8000 and you use `rangeBetween(-1000, 1000)`, it includes all salaries between 7000 and 9000, regardless of how many rows that includes from the current partition.

#### Window.currentRow
- Represents current row

#### Window.unboundedPreceding
- Includes all rows from the start of the partition.

#### Window.unboundedFollowing
- Extends the window to the last row of the partition.



[Reference](https://medium.com/@sachan.pratiksha/understanding-window-functions-in-sql-and-spark-with-rowbetween-rangebetween-unbounded-preceding-6cb2fd50dce6)

rowsBetween

In [12]:
# Calculate moving sum of salary considering 1 row before and 1 row after current row
windowSpec1 = Window.partitionBy("Department").orderBy("Salary").rowsBetween(-1, 1)
df_rowbetween = df.withColumn("moving_sum_salary", sum("Salary").over(windowSpec1))
df_rowbetween.select("Name", "Department", "Salary", "moving_sum_salary").show()

+------+----------+------+-----------------+
|  Name|Department|Salary|moving_sum_salary|
+------+----------+------+-----------------+
|   Ram|   Finance|  7900|            16100|
| Satya|   Finance|  8200|            24400|
| Sugie|   Finance|  8300|            25500|
|  Lisa|   Finance|  9000|            27200|
|  Deja|   Finance|  9900|            18900|
|  Kyle| Marketing|  8000|            17100|
|  Reid| Marketing|  9100|            17100|
| Kathy|     Sales|  1000|             9100|
|Robert|     Sales|  8100|            17200|
|  Ross|     Sales|  8100|            24800|
|Alicia|     Sales|  8600|            25300|
|  John|     Sales|  8600|            26200|
| James|     Sales|  9000|            17600|
+------+----------+------+-----------------+



rangeBetween

In [13]:
# Sum salaries within a range of 1000 (salary values within +/- 1000 of current row's salary)
windowSpec2 = Window.partitionBy("Department").orderBy("Salary").rangeBetween(-1000, 1000)
df_rangebetween = df.withColumn("salary_range_sum", sum("Salary").over(windowSpec2))
df_rangebetween.select("Name", "Department", "Salary", "salary_range_sum").show()

+------+----------+------+----------------+
|  Name|Department|Salary|salary_range_sum|
+------+----------+------+----------------+
|   Ram|   Finance|  7900|           24400|
| Satya|   Finance|  8200|           33400|
| Sugie|   Finance|  8300|           33400|
|  Lisa|   Finance|  9000|           35400|
|  Deja|   Finance|  9900|           18900|
|  Kyle| Marketing|  8000|            8000|
|  Reid| Marketing|  9100|            9100|
| Kathy|     Sales|  1000|            1000|
|Robert|     Sales|  8100|           42400|
|  Ross|     Sales|  8100|           42400|
|Alicia|     Sales|  8600|           42400|
|  John|     Sales|  8600|           42400|
| James|     Sales|  9000|           42400|
+------+----------+------+----------------+



Window.unboundedPreceding

In [14]:
# Calculate cumulative sum from start of partition to current row
windowSpec3 = Window.partitionBy("Department").orderBy("Salary").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_unbounded_preceding = df.withColumn("cumulative_salary", sum("Salary").over(windowSpec3))
df_unbounded_preceding.select("Name", "Department", "Salary", "cumulative_salary").show()

+------+----------+------+-----------------+
|  Name|Department|Salary|cumulative_salary|
+------+----------+------+-----------------+
|   Ram|   Finance|  7900|             7900|
| Satya|   Finance|  8200|            16100|
| Sugie|   Finance|  8300|            24400|
|  Lisa|   Finance|  9000|            33400|
|  Deja|   Finance|  9900|            43300|
|  Kyle| Marketing|  8000|             8000|
|  Reid| Marketing|  9100|            17100|
| Kathy|     Sales|  1000|             1000|
|Robert|     Sales|  8100|             9100|
|  Ross|     Sales|  8100|            17200|
|Alicia|     Sales|  8600|            25800|
|  John|     Sales|  8600|            34400|
| James|     Sales|  9000|            43400|
+------+----------+------+-----------------+



Window.unboundedFollowing

In [15]:
# Calculate sum from current row to end of partition
windowSpec4 = Window.partitionBy("Department").orderBy("Salary").rowsBetween(Window.currentRow, Window.unboundedFollowing)
df_unbounded_following = df.withColumn("future_salary_sum", sum("Salary").over(windowSpec4))
df_unbounded_following.select("Name", "Department", "Salary", "future_salary_sum").show()

+------+----------+------+-----------------+
|  Name|Department|Salary|future_salary_sum|
+------+----------+------+-----------------+
|   Ram|   Finance|  7900|            43300|
| Satya|   Finance|  8200|            35400|
| Sugie|   Finance|  8300|            27200|
|  Lisa|   Finance|  9000|            18900|
|  Deja|   Finance|  9900|             9900|
|  Kyle| Marketing|  8000|            17100|
|  Reid| Marketing|  9100|             9100|
| Kathy|     Sales|  1000|            43400|
|Robert|     Sales|  8100|            42400|
|  Ross|     Sales|  8100|            34300|
|Alicia|     Sales|  8600|            26200|
|  John|     Sales|  8600|            17600|
| James|     Sales|  9000|             9000|
+------+----------+------+-----------------+

